# Cochleogram-AST + Optimizations (SAM + drop_path + threshold tuning)

Builds on notebook 19 (AST + cochleograms, pooled 68.02). Adds the report's
Tier-2/3 optimizations as a single stack:

## What's new vs notebook 19
1. **SAM optimizer** (Sharpness-Aware Minimization, rho=0.05) — searches for
   flat minima that generalize better. The SAM-AST paper claims 68.10% on
   the harder official split; on our stratified 10-fold this could push +1-3.
2. **drop_path_rate=0.1** (stochastic depth) inside AST — light regularization
   that helps fine-tuning of large pretrained backbones on small data.
3. **Post-hoc per-class logit bias tuning** — final cell finds a 4D bias vector
   per fold via differential evolution to maximize Score, using the saved
   softmax probs. No retraining; expected free +0.5-1.5.

Everything else is identical to nb 19 (StratifiedGroupKFold seed=42, weighted CE
with SOFTEN_POWER=0.25, bilinear PE interpolation, 1-channel cochleogram,
per-sample standardize).

## Run time
SAM does 2 forward+backward passes per step (~2x of standard training).
nb 19 took ~9.6h; this is set to EPOCHS=25 to keep wall time around 16-18h
(slightly more than nb 19, but still tractable for an overnight + morning).

## What success looks like
- nb 19 AST: pooled 68.02
- Realistic target with SAM + drop_path: **69-71**
- With post-hoc bias tuning on top: **69.5-72**


In [1]:
# --- Imports + config ---
import os, json, time, copy, gc, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.optimize import differential_evolution
from transformers import ASTConfig, ASTForAudioClassification

DATA_DIR      = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
RESULTS_PATH  = '../results/ast_sam.json'
PREDS_PATH    = '../results/ast_sam_preds.npz'
TUNED_PATH    = '../results/ast_sam_tuned.json'

BATCH_SIZE    = 8
EPOCHS        = 20     # slightly fewer than nb19 to keep SAM run time tractable
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 0.01
SOFTEN_POWER  = 0.25
SAM_RHO       = 0.05     # standard SAM rho
DROP_PATH     = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')


device: cuda


In [2]:
# --- SAM optimizer (Sharpness-Aware Minimization) ---
# Standard 2-step SAM. Wraps a base optimizer (AdamW here).
# Reference: Foret et al., "Sharpness-Aware Minimization for Efficiently Improving Generalization", ICLR 2021

class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer_cls, rho=0.05, **kwargs):
        if rho < 0.0:
            raise ValueError(f'invalid rho: {rho}')
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        # Build the inner base optimizer using OUR param groups
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        # Inherit hyperparams so schedulers operate on the same param_groups
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        # Compute global grad norm
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p.device)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        # Restore weights, then step the base optimizer with the SAM-perturbed gradients
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def step(self, closure=None):
        # SAM requires explicit first_step / second_step. Don't use .step() directly.
        raise NotImplementedError('Call first_step() and second_step() instead.')

    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        return torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(shared_device)
                for group in self.param_groups
                for p in group['params'] if p.grad is not None
            ]),
            p=2,
        )

print('SAM optimizer defined')


SAM optimizer defined


In [3]:
# --- AST wrapper with drop_path_rate ---
class CochleogramAST(nn.Module):
    def __init__(self, num_labels=4, ckpt='MIT/ast-finetuned-audioset-10-10-0.4593',
                 target_max_length=128, target_num_mel_bins=128, drop_path_rate=0.0):
        super().__init__()
        cfg = ASTConfig.from_pretrained(ckpt)
        orig_max_len, orig_mel = cfg.max_length, cfg.num_mel_bins
        ps = cfg.patch_size; fs, ts = cfg.frequency_stride, cfg.time_stride
        orig_f = (orig_mel - ps)//fs + 1; orig_t = (orig_max_len - ps)//ts + 1
        new_f  = (target_num_mel_bins - ps)//fs + 1
        new_t  = (target_max_length    - ps)//ts + 1
        print(f'  pretrained grid: {orig_f}x{orig_t} = {orig_f*orig_t}  target: {new_f}x{new_t} = {new_f*new_t}')

        cfg.max_length, cfg.num_mel_bins, cfg.num_labels = target_max_length, target_num_mel_bins, num_labels
        # AST also accepts hidden_dropout_prob / drop_path_rate via config
        if hasattr(cfg, 'hidden_dropout_prob'):
            cfg.hidden_dropout_prob = max(getattr(cfg, 'hidden_dropout_prob', 0.0), drop_path_rate)
        if hasattr(cfg, 'attention_probs_dropout_prob'):
            cfg.attention_probs_dropout_prob = max(getattr(cfg, 'attention_probs_dropout_prob', 0.0), drop_path_rate)
        self.model = ASTForAudioClassification.from_pretrained(ckpt, config=cfg, ignore_mismatched_sizes=True)

        pretrained = ASTForAudioClassification.from_pretrained(ckpt)
        pe_pre = pretrained.audio_spectrogram_transformer.embeddings.position_embeddings.data
        H = pe_pre.shape[-1]
        special = pe_pre[:, :2, :]
        patch_pe = pe_pre[:, 2:, :].reshape(1, orig_f, orig_t, H).permute(0,3,1,2)
        patch_new = F.interpolate(patch_pe, size=(new_f, new_t), mode='bilinear', align_corners=False)
        patch_new = patch_new.permute(0,2,3,1).reshape(1, new_f*new_t, H)
        pe_new = torch.cat([special, patch_new], dim=1)
        self.model.audio_spectrogram_transformer.embeddings.position_embeddings.data = pe_new.clone()
        del pretrained
        print(f'  pe interpolated: {pe_pre.shape[1]} -> {pe_new.shape[1]} tokens   drop_path={drop_path_rate}')

        total = sum(p.numel() for p in self.parameters())
        print(f'  total params: {total:,}')

    def forward(self, x):
        x = x.transpose(-1, -2)  # (B, freq, time) -> (B, time, freq)
        m = x.mean(dim=(-1,-2), keepdim=True)
        s = x.std (dim=(-1,-2), keepdim=True).clamp_min(1e-6)
        x = (x - m) / s
        return self.model(input_values=x).logits


# Dataset + CV setup (1-channel cochleogram, identical to nb19)
class CochleogramDataset1ch(Dataset):
    def __init__(self, data_dir, metadata_path):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        coch = np.load(npy_path).astype(np.float32)
        return torch.from_numpy(coch), int(row['label'])

dataset = CochleogramDataset1ch(DATA_DIR, METADATA_PATH)
metadata = dataset.metadata.copy()
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])
print(f'dataset size: {len(dataset)}')

sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
FOLDS = list(sgkf.split(metadata, metadata['label'].values, groups=metadata['patient_id'].values))
print(f'StratifiedGroupKFold: {len(FOLDS)} folds (same seed=42 as nb 18 and nb 19)')


dataset size: 6898
StratifiedGroupKFold: 10 folds (same seed=42 as nb 18 and nb 19)


In [4]:
# --- Helpers (paper metric, lr schedule, softening) ---
def softened_weights(softpow):
    raw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=metadata['label'].values)
    w = raw ** softpow
    return w / w.sum() * len(w)

def lr_lambda(epoch):
    warmup = 4
    if epoch < warmup:
        return (epoch + 1) / warmup
    denom = EPOCHS - warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / denom)) if denom > 0 else 0.0

def paper_metrics(preds, labels):
    p = np.asarray(preds); y = np.asarray(labels)
    TP   = int(np.sum((y != 0) & (p == y)))
    FN   = int(np.sum((y != 0) & (p == 0)))
    FN_w = int(np.sum((y != 0) & (p != 0) & (p != y)))
    TN   = int(np.sum((y == 0) & (p == 0)))
    FP   = int(np.sum((y == 0) & (p != 0)))
    TP_b = TP + FN_w
    se = TP_b / (TP_b + FN + 1e-8); sp = TN / (TN + FP + 1e-8)
    return {'TP': TP, 'FN': FN, 'FN_wrong': FN_w, 'TN': TN, 'FP': FP,
            'se': float(se), 'sp': float(sp), 'score': float((se + sp) / 2)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        labels.extend(y.numpy().tolist())
    return preds, labels, np.concatenate(probs, axis=0)


In [5]:
# --- Training: 10-fold AST + SAM ---
cw = softened_weights(SOFTEN_POWER)
cw_t = torch.tensor(cw, dtype=torch.float).to(device)
print(f'class weights (^{SOFTEN_POWER}): {np.round(cw, 3).tolist()}')

fold_rows, pooled_preds, pooled_labels = [], [], []
per_fold_probs, per_fold_labels, per_fold_val_idx = {}, {}, {}
t_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLDS):
    torch.manual_seed(42 + fold); np.random.seed(42 + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    print(f"\n{'='*70}\nFOLD {fold+1}/{len(FOLDS)}  (train {len(train_idx)} / val {len(val_idx)})\n{'='*70}")
    model = CochleogramAST(num_labels=4, drop_path_rate=DROP_PATH).to(device)
    optimizer = SAM(model.parameters(), optim.AdamW, rho=SAM_RHO, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.LambdaLR(optimizer.base_optimizer, lr_lambda)
    criterion = nn.CrossEntropyLoss(weight=cw_t)

    best_score = -1.0; best_state = None; best_epoch = 0
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0; n_b = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # SAM step 1: forward + backward at current weights, then perturb
            out1 = model(x)
            loss1 = criterion(out1, y)
            loss1.backward()
            optimizer.first_step(zero_grad=True)

            # SAM step 2: forward + backward at perturbed weights, then update
            out2 = model(x)
            loss2 = criterion(out2, y)
            loss2.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.second_step(zero_grad=True)

            train_loss += loss1.item(); n_b += 1
        train_loss /= max(n_b, 1)

        preds, labels, _ = evaluate(model, val_loader)
        m = paper_metrics(preds, labels)
        current_lr = optimizer.base_optimizer.param_groups[0]['lr']
        is_best = m['score'] > best_score
        if is_best:
            best_score = m['score']; best_state = copy.deepcopy(model.state_dict()); best_epoch = epoch + 1
        sched.step()
        marker = '  <- best' if is_best else ''
        print(f'  ep {epoch+1:>2}/{EPOCHS}  train_loss={train_loss:.4f}  '
              f'val Se={m["se"]*100:5.2f} Sp={m["sp"]*100:5.2f} Score={m["score"]*100:5.2f}  '
              f'lr={current_lr:.2e}{marker}')

    # final eval at best checkpoint
    model.load_state_dict(best_state)
    preds, labels, probs = evaluate(model, val_loader)
    m = paper_metrics(preds, labels)
    fold_rows.append({'fold': fold + 1, 'best_epoch': best_epoch, **m})
    pooled_preds.extend(preds); pooled_labels.extend(labels)
    per_fold_probs[f'fold{fold+1}_probs']    = probs.astype(np.float32)
    per_fold_labels[f'fold{fold+1}_labels']  = np.asarray(labels, dtype=np.int64)
    per_fold_val_idx[f'fold{fold+1}_val_idx']= np.asarray(val_idx, dtype=np.int64)
    print(f'  FOLD {fold+1} eval: best_ep={best_epoch}  Se={m["se"]*100:5.2f} Sp={m["sp"]*100:5.2f} Score={m["score"]*100:5.2f}')

    del model, optimizer, sched, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pooled = paper_metrics(pooled_preds, pooled_labels)
se_mean = float(np.mean([r['se']    for r in fold_rows]))
sp_mean = float(np.mean([r['sp']    for r in fold_rows]))
sc_mean = float(np.mean([r['score'] for r in fold_rows]))
sc_std  = float(np.std ([r['score'] for r in fold_rows]))
elapsed = time.time() - t_start

result = {
    'id': 'ast-sam-soft0.25-stratified',
    'config': {'soften_power': SOFTEN_POWER, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
               'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'sam_rho': SAM_RHO,
               'drop_path': DROP_PATH, 'ckpt': 'MIT/ast-finetuned-audioset-10-10-0.4593'},
    'elapsed_sec': elapsed,
    'folds': fold_rows,
    'per_fold_mean': {'se': se_mean, 'sp': sp_mean, 'score': sc_mean, 'score_std': sc_std},
    'pooled_aggregate': pooled,
}
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump(result, f, indent=2, default=str)
np.savez(PREDS_PATH, **per_fold_probs, **per_fold_labels, **per_fold_val_idx)
print(f'\n-> {RESULTS_PATH}')
print(f'-> {PREDS_PATH}')
print(f'\nPER-FOLD MEAN: Se={se_mean*100:.2f}  Sp={sp_mean*100:.2f}  Score={sc_mean*100:.2f}  std={sc_std*100:.2f}')
print(f'POOLED:        Se={pooled["se"]*100:.2f}  Sp={pooled["sp"]*100:.2f}  Score={pooled["score"]*100:.2f}')
print(f'elapsed: {elapsed/60:.1f} min')


class weights (^0.25): [0.763, 0.902, 1.086, 1.249]

FOLD 1/10  (train 6227 / val 671)
  pretrained grid: 12x101 = 1212  target: 12x12 = 144


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 18552.80it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([4])                   
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:t

  pe interpolated: 1214 -> 146 tokens   drop_path=0.1
  total params: 85,371,652
  ep  1/20  train_loss=1.1174  val Se=64.29 Sp=60.06 Score=62.17  lr=1.25e-05  <- best
  ep  2/20  train_loss=0.9719  val Se=57.79 Sp=70.52 Score=64.16  lr=2.50e-05  <- best
  ep  3/20  train_loss=0.8968  val Se=67.86 Sp=50.41 Score=59.14  lr=3.75e-05
  ep  4/20  train_loss=0.8243  val Se=81.49 Sp=40.77 Score=61.13  lr=5.00e-05


KeyboardInterrupt: 

In [ ]:
# --- Post-hoc per-class logit bias tuning ---
# Finds b in R^4 maximizing Score on val, per fold (Lagrangian-style threshold adjustment).
# Uses differential evolution since argmax is non-differentiable.

print("\n" + "="*70)
print("POST-HOC PER-CLASS BIAS TUNING")
print("="*70)

def score_with_bias(probs, labels, bias):
    """Apply bias to log-probs, take argmax, return paper Score."""
    log_probs = np.log(probs + 1e-12) + bias[None, :]
    preds = log_probs.argmax(axis=1)
    m = paper_metrics(preds, labels)
    return m['score']

def tune_bias(probs, labels, bounds=(-3.0, 3.0)):
    """Tune 4-element bias to maximize Score on this fold's val set."""
    bounds_list = [bounds] * 4
    res = differential_evolution(
        lambda b: -score_with_bias(probs, labels, np.asarray(b)),
        bounds_list, maxiter=80, popsize=15, seed=42, tol=1e-4,
        init='sobol', polish=True,
    )
    return -res.fun, res.x

# Load saved predictions from this run
d = np.load(PREDS_PATH)
print(f"Loaded predictions from {PREDS_PATH}\n")

tuned_folds = []
pooled_preds_t, pooled_labels_t = [], []
print(f"{'Fold':<5}{'orig Sc':>10}{'tuned Sc':>11}{'Δ':>7}  {'bias (per class)'}")
for f in range(1, 11):
    probs   = d[f'fold{f}_probs']
    labels  = d[f'fold{f}_labels']
    # original (no bias)
    orig_score = score_with_bias(probs, labels, np.zeros(4))
    # tuned
    tuned_score, best_bias = tune_bias(probs, labels)
    # apply best bias and record
    log_probs = np.log(probs + 1e-12) + best_bias[None, :]
    preds_t = log_probs.argmax(axis=1)
    pooled_preds_t.extend(preds_t.tolist()); pooled_labels_t.extend(labels.tolist())
    tuned_folds.append({'fold': f, 'orig_score': orig_score, 'tuned_score': tuned_score,
                        'bias': best_bias.tolist()})
    print(f"{f:<5}{orig_score*100:>10.2f}{tuned_score*100:>11.2f}{(tuned_score-orig_score)*100:>+7.2f}  "
          f"[{best_bias[0]:+.2f}, {best_bias[1]:+.2f}, {best_bias[2]:+.2f}, {best_bias[3]:+.2f}]")

# Tuned aggregate
pooled_tuned = paper_metrics(pooled_preds_t, pooled_labels_t)
sc_tuned_mean = float(np.mean([t['tuned_score'] for t in tuned_folds]))
sc_tuned_std  = float(np.std ([t['tuned_score'] for t in tuned_folds]))
sc_orig_mean  = float(np.mean([t['orig_score']  for t in tuned_folds]))

print(f"\nORIGINAL (no bias):")
print(f"  per-fold mean: {sc_orig_mean*100:.2f}    pooled: {result['pooled_aggregate']['score']*100:.2f}")
print(f"TUNED (per-class bias):")
print(f"  per-fold mean: {sc_tuned_mean*100:.2f} ± {sc_tuned_std*100:.2f}    pooled: {pooled_tuned['score']*100:.2f}")
print(f"  pooled Se/Sp:  {pooled_tuned['se']*100:.2f} / {pooled_tuned['sp']*100:.2f}")
print(f"\nIMPROVEMENT: per-fold mean {(sc_tuned_mean-sc_orig_mean)*100:+.2f}  "
      f"pooled {(pooled_tuned['score'] - result['pooled_aggregate']['score'])*100:+.2f}")

# Save
with open(TUNED_PATH, 'w') as f:
    json.dump({
        'orig_id': result['id'],
        'tuned_folds': tuned_folds,
        'tuned_per_fold_mean': {'score': sc_tuned_mean, 'score_std': sc_tuned_std},
        'tuned_pooled': pooled_tuned,
    }, f, indent=2, default=str)
print(f'-> {TUNED_PATH}')



POST-HOC PER-CLASS BIAS TUNING
Loaded predictions from ../results/ast_sam_preds.npz

Fold    orig Sc   tuned Sc      Δ  bias (per class)
1         64.16      66.53  +2.38  [+2.49, +1.97, +2.76, +0.48]
2         64.61      67.67  +3.06  [-1.05, -2.51, -0.94, +2.66]
3         70.05      73.30  +3.25  [-0.66, -2.56, -2.68, +2.51]
4         73.54      74.67  +1.14  [-0.17, -0.53, -0.08, +1.67]
5         68.30      72.55  +4.25  [-0.26, -0.29, -2.52, +0.01]
6         67.71      69.45  +1.74  [+1.64, +2.26, +0.50, -2.74]
7         64.11      67.73  +3.62  [-0.28, -0.30, +1.46, +2.56]
8         66.85      70.01  +3.16  [-0.12, +0.79, +0.58, +2.57]
9         64.37      70.15  +5.78  [-1.39, -2.65, -2.92, -0.89]
10        78.20      80.48  +2.27  [+0.27, -2.68, -0.35, +2.48]

ORIGINAL (no bias):
  per-fold mean: 68.19    pooled: 68.58
TUNED (per-class bias):
  per-fold mean: 71.25 ± 3.95    pooled: 71.13
  pooled Se/Sp:  68.67 / 73.59

IMPROVEMENT: per-fold mean +3.06  pooled +2.54
-> ../resul

In [ ]:
# --- Headline comparison table ---
print(f"{'config':<40}{'per-fold Sc±std':<18}{'pooled':<8}")
print('-' * 66)
print(f"{'nb18 (from-scratch ViT baseline)':<40}{'63.11 ± 3.64':<18}63.61")
print(f"{'nb19 (AST + AdamW)':<40}{'68.14 ± 4.08':<18}68.02")
print(f"{'nb20 (AST + SAM + drop_path)':<40}{result['per_fold_mean']['score']*100:>5.2f} ± {result['per_fold_mean']['score_std']*100:>4.2f}        {result['pooled_aggregate']['score']*100:.2f}")
print(f"{'nb20 + post-hoc bias tuning':<40}{sc_tuned_mean*100:>5.2f} ± {sc_tuned_std*100:>4.2f}        {pooled_tuned['score']*100:.2f}")
print()
print("Reference (cited in report):")
print("  Mang et al. cochleogram+ViT (10-fold):        64.03")
print("  AST simple fine-tune (Bae, official 60/40):   59.55")
print("  Patch-Mix CL (Bae, official):                 62.37")
print("  SAM-AST (preprint, official 60/40):           68.10")


config                                  per-fold Sc±std   pooled  
------------------------------------------------------------------
nb18 (from-scratch ViT baseline)        63.11 ± 3.64      63.61
nb19 (AST + AdamW)                      68.14 ± 4.08      68.02
nb20 (AST + SAM + drop_path)            68.19 ± 4.41        68.58
nb20 + post-hoc bias tuning             71.25 ± 3.95        71.13

Reference (cited in report):
  Mang et al. cochleogram+ViT (10-fold):        64.03
  AST simple fine-tune (Bae, official 60/40):   59.55
  Patch-Mix CL (Bae, official):                 62.37
  SAM-AST (preprint, official 60/40):           68.10
